# Measuring tumor compartments: overlap is not volume

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/01_imaging_measurements.ipynb)

**Question:** Can two contours have the same volume but different overlap, and does
this matter for a real segmentation model?
**Evidence:** A constructed ellipsoid on a millimetre grid (Figure 1); the published
study's illustration (Figure 2); and a recomputation of dataset-reference versus model
masks in its 50-subject cohort (Figure 3), from a local companion checkout or the released
cohort archive. Scientific functions live in glioma_ai.
Run this notebook with the uv kernel documented in README.md. In Google Colab, the first code cell clones the public repository and puts its package on the path; Colab's own NumPy, Matplotlib and nibabel are used, not the locked versions.

**Place in the perspective:** Start with notebook 09 for our published imaging foundation, methods and attributed results. Synthetic, published and locally recomputed evidence stay separately labeled here.

## AI contribution and verification

**Question.** When a tumor is outlined slightly differently, by a model, a second reader or the
same reader at another visit, how much do the numbers clinicians rely on change?
- *Volume* drives volumetric response assessment (RANO 2.0 optionally uses ≥ 40% increase for
  progression and ≥ 65% decrease for partial response, compared between examinations) and
  extent-of-resection estimates.
- *Location and overlap* drive regional burden near eloquent cortex, resection planning and
  radiotherapy target definition.
- Segmentation papers usually report Dice. This notebook asks what Dice does and does not tell a
  clinician about those two quantities, first in a construction where the answer is known
  (Figure 1), then in the published model's own evaluation cohort (Figure 3).

**AI contribution.** Three kinds of computation appear here, and they need different checks.
- *Trained imaging AI* (published study, not rerun): a MONAI BraTS SegResNet takes T1, T1c, T2 and
  FLAIR and labels necrotic core, edema and enhancing tumor. Figure 3 evaluates its stored label
  maps for 50 UCSF-PDGM subjects against the dataset's reference segmentations.
- *Generative research assistance* (development only; records in `docs/ai-use/`): adapting the
  unit-explicit measurement code, designing the synthetic construction and perturbations, the
  figure layouts and captions, the cohort recomputation module and its tests.
- *Conventional code* (rerun every time, no model): NIfTI reading, affine-based volumes and Dice.
  Rebuilding the notebook calls neither an LLM nor the segmentation model.

**Investigator choices.**
The investigator defined the scope: an imaging foundation for the perspective, explicitly labelled
figures with paper-style captions, and a real-data counterpart from the published study
(Lundervold et al. 2026) and its companion code, including its 50-subject evaluation cohort. Within
that scope, the design decisions and their reasons are:
- *Voxel grid* 1.2 × 1.2 × 2.0 mm: anisotropic, like many clinical acquisitions, so that units and
  slice spacing cannot be ignored.
- *Lesion size* (semi-axes 10, 12 and 16 mm; 8 mL whole tumor): a small lesion, where a one-voxel
  boundary error weighs most.
- *Perturbations*: a 4.8 mm shift, the order of a registration or repositioning error between
  examinations; and a 12% expansion, about one voxel at the boundary, chosen because it changes
  volume by ≈ 40%, the size of the RANO 2.0 volumetric progression threshold.
- *Label nesting*: necrotic core, enhancing rim and surrounding edema, as in typical glioblastoma.
- *Real-data case*: the subject closest to the cohort's median whole-tumor Dice, a representative
  rather than a best or worst case, shown on its largest-area slice so all compartments are visible.
- *RANO 2.0 thresholds*: drawn for scale only, because they define change between examinations,
  not segmentation error.

The clinical interpretations are teaching illustrations of measurement behaviour; individual cases
still require review by a neuroradiologist or neurosurgeon.

**Verification.**
- *Units and geometry*: volume = voxel count × |det A| of the NIfTI affine; NIfTI round trip within
  1e-5 mL; comparisons only on identical grids, with no silent registration or resampling; a copy
  whose affine alone is moved is refused (shown below).
- *Labels*: only 0, 1, 2 and 4 are accepted; WT = 1 + 2 + 4, TC = 1 + 4, ET = 4; the union of
  compartments equals the reference volume.
- *Against an independent expectation*: the voxelized 12% expansion (+41.1%) is compared with the
  continuous value 1.12³ − 1 = 40.5%.
- *Against the published study*: recomputed per-subject Dice must equal the companion's stored
  records, and the median Dice must agree with the published 0.90/0.94/0.86 at two decimals; either
  mismatch stops the analysis. Automated tests cover these checks with synthetic fixtures.
- *Not verified here*: accuracy of the reference labels themselves (they are the dataset's
  segmentations, not histology), reader or test–retest variability, and any link to treatment
  decisions or patient outcome.

**Evidence status.**
- *Figure 1*: synthetic construction. It shows what can happen to Dice and volume; it is not an
  estimate of AI or reader performance.
- *Figure 2*: attributed published illustration (CC BY 4.0), shown unmodified; anatomical context,
  not functional eloquence or clinical risk.
- *Figure 3*: local recomputation from existing dataset-reference and published-model masks. It
  describes agreement and systematic volume bias in one public cohort of IDH-wildtype glioblastoma;
  no inference is rerun, no new benchmark is claimed, and no response category, resection extent or
  outcome is assessed. Its per-subject values stay local and out of Git.

These roles describe development and provenance. [Assistance records](../docs/ai-workflow.md) retain the review status; record completeness is separate from scientific quality.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False


def repository_root():
    candidates = (Path.cwd(), *Path.cwd().parents)
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)


if IN_COLAB and repository_root() is None:
    checkout = Path("/content/AI-assisted-glioma-research")
    if not checkout.is_dir():
        url = "https://github.com/arvidl/AI-assisted-glioma-research"
        subprocess.run(["git", "clone", "--depth", "1", url, str(checkout)], check=True)
    pip = [sys.executable, "-m", "pip", "install", "--quiet"]
    subprocess.run([*pip, "nibabel>=5.3,<6"], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(Path.cwd() / "src"))
ROOT = repository_root()
if ROOT is None:
    raise RuntimeError("Open this notebook in a checkout of the repository or in Google Colab.")

## Our published imaging foundation

[Lundervold et al., Journal of Medical Engineering & Technology (2026)](https://doi.org/10.1080/03091902.2026.2713703)
combines multiparametric-MRI segmentation with subject-specific anatomical
parcellation. Compartment-region intersections produce descriptive regional
burden **hit-plots**. [The companion code](https://github.com/arvidl/high-precision-glioma-segmentation)
is the full imaging workflow; this notebook does not run it.

The next cell reads checked **published aggregate values**, not measurements
from the synthetic construction. Whole tumor (WT), tumor core (TC), and
enhancing tumor (ET) are evaluated regions, distinct from the mutually
exclusive labels defined below. The reported population is glioblastoma,
IDH-wildtype, WHO grade 4; the study is a feasibility report without a
clinical-utility or patient-outcome evaluation.

An anatomical profile can locate a biological research question. It does not
identify cell programs, neuron-glioma synapses, or functional eloquence.
[The source record](../docs/related-work/high-precision-glioma-segmentation.md)
retains provenance and these interpretation boundaries.

In [ ]:
from glioma_ai.published import published_imaging_reference

published = published_imaging_reference()
print("Published source:", published["doi"])
print("Pipeline rerun here:", published["inference_rerun_in_this_repository"])
published["evaluation"]

## Construction and explicit definitions
A constructed ellipsoid (semi-axes 10, 12 and 16 mm) sits on a 64 × 64 × 40 grid
with 1.2 × 1.2 × 2.0 mm voxels. Three nested, mutually exclusive labels follow the
BraTS numbering and the typical glioblastoma arrangement: label 1, necrotic core or
non-enhancing tumor (innermost); label 4, an enhancing rim; label 2, edema/FLAIR
abnormality (outermost). They are prescribed, not tissue observations or model output.
Two perturbations of the whole tumor (labels 1 + 2 + 4) are compared with it: a
translation by 4 voxels (4.8 mm) and a 12% increase of every semi-axis.

In [ ]:
from glioma_ai.demo import imaging_example

OUTPUT = ROOT / "outputs/notebook_01"
result = imaging_example(OUTPUT)
display(Image(filename=str(OUTPUT / "imaging.png")))
print(f"Reference whole-tumor volume: {result['reference_volume_ml']:.3f} mL")
print(
    f"Translated {result['translation_mm']:.1f} mm: "
    f"Dice {result['translated_dice']:.3f}, volume error 0.0%"
)
print(
    f"Expanded {result['radius_expansion_percent']}%: Dice {result['expanded_dice']:.3f}, "
    f"volume error {100 * result['expanded_relative_volume_error']:+.1f}%"
)
print("Volume by label (mL):", {k: round(v, 3) for k, v in result["volume_by_label_ml"].items()})

**Figure 1. Overlap and volume respond differently to two small segmentation errors (synthetic construction).**
**(A)** Mid-plane (z = +1 mm from the ellipsoid centre) of a constructed tumor on a 64 × 64 × 40 grid of
1.2 × 1.2 × 2.0 mm voxels. Three nested, mutually exclusive labels follow the BraTS numbering and the
typical glioblastoma arrangement: label 1, necrotic core or non-enhancing tumor (purple; 0.53 mL);
label 4, enhancing rim (orange; 2.17 mL); label 2, edema/FLAIR abnormality (teal; 5.32 mL). The labels
are prescribed regions, not observed tissue or model output.
**(B)** The reference whole tumor (labels 1 + 2 + 4; grey; 8.02 mL) with two perturbed contours: the same
mask translated by 4 voxels (4.8 mm) along x (magenta), and an ellipsoid with every semi-axis enlarged
by 12% (blue; 1.2–1.9 mm, about one voxel at the boundary).
**(C)** Whole-tumor volume of each mask, annotated with the signed volume error ΔV relative to the
reference and the Dice overlap with it. The translation preserves volume exactly (ΔV 0.0%) but lowers
Dice to 0.644. The expansion keeps Dice at 0.830, a value often read as good agreement, yet overestimates
volume by 41.1% (continuous expectation 1.12³ − 1 = 40.5%; the difference is voxelization).
Volumes are voxel counts multiplied by |det A| of the NIfTI affine; Dice is computed on the identical
grid without registration or resampling. No patient data or trained model is involved.

## Check against an independent measurement
The union of selected compartments must equal the binary reference volume.
Volume uses the determinant of the spatial affine, including rotations and shear.
The stored NIfTI-1 affine uses finite precision; the volume comparison allows 1e-5 mL of serialization tolerance.

In [ ]:
from glioma_ai.imaging import summarize_labels

measurement = summarize_labels(OUTPUT / "synthetic_labels.nii.gz", [1, 2, 4])
assert abs(measurement["selected_volume_ml"] - result["reference_volume_ml"]) < 1e-5
assert result["translated_dice"] < 1
measurement

## Same shape is not the same space
Two label maps are comparable only on the same physical grid. Move only the
affine of a copy of the reference by 10 mm: the arrays keep their shape and
values, yet the comparison must be refused rather than silently registered
or resampled.

In [ ]:
import nibabel as nib

from glioma_ai.imaging import load_label_map

values, affine_mm = load_label_map(OUTPUT / "synthetic_reference.nii.gz")
moved = affine_mm.copy()
moved[0, 3] += 10
image = nib.Nifti1Image(values, moved)
image.header.set_xyzt_units("mm")
nib.save(image, OUTPUT / "synthetic_moved.nii.gz")
try:
    summarize_labels(
        OUTPUT / "synthetic_moved.nii.gz",
        [1],
        OUTPUT / "synthetic_reference.nii.gz",
    )
except ValueError as error:
    print("Expected geometry refusal:", error)
else:
    raise AssertionError("Different physical grids were accepted")

Report every volume with its coordinate space: a voxel count after deformable
warping to an atlas is not automatically a native-space tumor volume.
Exercise: summarize labels 1 and 4 of `synthetic_labels.nii.gz`. Which
compartment is excluded? For a real file, obtain its label definitions and
provenance before measuring.

## What Figure 1 teaches
Dice and volume measure different failures, and neither implies the other.

- **A location error keeps the volume and loses the overlap.** A 4.8 mm shift is the size
  of a plausible registration or repositioning error between examinations or MRI sequences.
  Volume cannot see it; Dice and distance measures can. Location matters for regional
  burden and hit-plots (the published study), radiotherapy targets and resection planning
  near eloquent cortex.
- **A size error keeps the overlap high and inflates the volume.** Moving the boundary about
  one voxel outward changes volume by roughly three times the radial fraction (1.12³ ≈ 1.40),
  while Dice stays at 0.83. That volume change is the size of the optional RANO 2.0
  volumetric progression threshold (+40%; Wen et al. 2023), although RANO concerns change
  between examinations, not segmentation error.
- **Report both, with units and conventions.** State the voxel spacing, label convention,
  evaluated region (WT, TC, ET) and whether the comparison grids are identical.

Neither perturbation is an AI performance estimate. Exercise: compute the continuous volume
multiplier for a 12% radius expansion and compare it with the voxelized multiplier below.
Why need they differ?

In [ ]:
continuous_multiplier = 1.12**3
voxelized_multiplier = result["expanded_volume_ml"] / result["reference_volume_ml"]
{"continuous_multiplier": continuous_multiplier, "voxelized_multiplier": voxelized_multiplier}

## Real-data counterpart: the published study
The same question applies to the published segmentation study
([Lundervold et al. 2026](https://doi.org/10.1080/03091902.2026.2713703)), which reports median Dice
0.90 (WT), 0.94 (TC) and 0.86 (ET) against dataset reference masks in 50 UCSF-PDGM glioblastoma
subjects. **Figure 2** is the published regional-anatomy illustration, displayed unmodified when it has
been imported locally (`make published-figures` from the author checkout, or
`make published-figures-fetch` from the release `published-figures-v1`; in Colab the next cell
fetches it). **Figure 3** asks what Dice alone does not report:
are the model's errors mainly shifts or systematic size differences?

**Google Colab only.** When you run the next cell in Colab, it downloads the three unmodified
published figures (CC BY 4.0, Lundervold et al. 2026) from the release `published-figures-v1`,
checks each against the SHA-256 pinned in `src/glioma_ai/references/imaging_figures.json` and
writes them to `paper/generated/`. Outside Colab the cell changes nothing.

In [ ]:
from glioma_ai.published import fetch_published_figures, prepared_published_figures

FIGURES = ROOT / "paper/generated"
if not IN_COLAB:
    print("Not Colab: figures come from make published-figures or make published-figures-fetch.")
elif all(figure["ready"] for figure in prepared_published_figures(FIGURES)):
    print("Colab: using the published figures already present.")
else:
    try:
        fetched = fetch_published_figures(FIGURES)
    except FileNotFoundError as error:
        print(f"Colab: published figures not fetched, so Figure 2 links to the article.\n{error}")
    else:
        print(f"Colab: {fetched['figures']} figures verified; {fetched['license']}.")

In [ ]:
from IPython.display import Markdown

from glioma_ai.published import prepared_published_figures

anatomy = next(
    figure
    for figure in prepared_published_figures(ROOT / "paper/generated")
    if figure["name"] == "hpgs_regional_anatomy.png"
)
if anatomy["ready"]:
    display(Image(filename=anatomy["path"]))
else:
    display(
        Markdown(
            f"Figure not imported locally; see the [published article]({anatomy['source_url']})."
        )
    )

**Figure 2. Published regional-anatomy illustration (Lundervold et al. 2026, Figure 4a–b; CC BY 4.0; original author asset, unmodified).**
Public subject UCSF-PDGM-0020. **(a)** Sagittal, coronal and axial post-contrast T1-weighted slices
through the tumor with dataset-reference compartment contours (green) in the subject's own space.
**(b)** The same slices with the subject-specific anatomical parcellation, labelling the regions the tumor
contours intersect, such as left supramarginal, precentral, postcentral and superior temporal cortex and
white matter. Regional burden is computed from these compartment–region intersections (the published
hit-plot, Figure 4c, in notebook 09). This is the location-dependent use case for Figure 1B: a boundary
shift of a few millimetres can change which named regions are touched, while total volume stays
the same. The illustration shows anatomical context; it does not establish functional eloquence or
clinical risk.

## Figure 3: recomputation in the published cohort
This cell reads, without copying, the dataset-provided UCSF-PDGM reference segmentations and the
SegResNet label maps the companion study produced for its 50-subject evaluation cohort. It recomputes
Dice and **signed** volume error for WT, TC and ET, checks Dice against the companion's per-subject
records, and checks the medians against the publication. Grids must be identical and labels must be
0, 1, 2 or 4; nothing is registered, resampled or relabeled. No model is run.

It needs the staged cohort: a local companion checkout, or the released cohort archive
(`make hpgs-bundle-fetch`, 9 MB, release `ucsf-pdgm-cohort50-v1`), named by `HPGS_REPO` or in the
ignored `local/hpgs.json` as `{"checkout": "/absolute/path"}`. In Colab the next cell fetches the
archive. UCSF-PDGM v5 (Calabrese et al. 2022; The Cancer Imaging Archive,
[doi:10.7937/tcia.bdgf-8v37](https://doi.org/10.7937/tcia.bdgf-8v37)) is licensed CC BY 4.0, and
the TCIA Data Usage Policy applies; the model label maps are the companion study's (Lundervold et
al. 2026). The figure shows one public subject with this attribution. Without inputs the cell
prints a notice.

**Google Colab only.** When you run the next cell in Colab, it downloads the released cohort
inputs of Figure 3 (`ucsf_pdgm_cohort50_v1.zip`, 9 MB: 50 reference segmentations, 50 model label
maps with their records, the companion's metrics records and one T1c), checks the pinned SHA-256,
extracts them to `data/` and verifies every file against the archive's manifest. Outside Colab the
cell changes nothing ([data guide](../docs/data.md)).

In [ ]:
from glioma_ai.hpgs_cohort import companion_checkout

if not IN_COLAB:
    print("Not Colab: inputs come from HPGS_REPO or local/hpgs.json.")
elif companion_checkout(ROOT) is not None:
    print("Colab: using the configured cohort inputs.")
else:
    from glioma_ai.hpgs_bundle import fetch_bundle as fetch_cohort

    try:
        fetched = fetch_cohort(ROOT / "data")
    except FileNotFoundError as error:
        print(f"Colab: cohort inputs not fetched, so Figure 3 is skipped.\n{error}")
    else:
        os.environ["HPGS_REPO"] = fetched["checkout"]
        print(f"Colab: {fetched['files']} files verified; {fetched['use_condition']}.")

In [ ]:
from glioma_ai.hpgs_cohort import (
    cohort_caption,
    cohort_table,
    companion_checkout,
    write_cohort_figure,
)

checkout = companion_checkout(ROOT)
if checkout is None:
    display(
        Markdown(
            "**Figure 3 skipped:** no cohort inputs configured (`HPGS_REPO` or "
            "`local/hpgs.json`; `make hpgs-bundle-fetch` downloads them)."
        )
    )
else:
    cohort = write_cohort_figure(checkout, cohort_table(checkout), result, OUTPUT)
    display(Image(filename=str(OUTPUT / "hpgs_cohort.png")))
    display(Markdown(cohort_caption(cohort)))

## What the real data add
Read Figure 3 together with Figure 1:

- **Where do the cohort points lie?** Points near zero volume error with lower Dice behave like the
  synthetic shift; points far below or above zero behave like a size error. A cloud mostly below zero
  means the model draws boundaries systematically inside the reference, which a median Dice cannot show.
- **Which region is most affected?** Compare WT, TC and ET. A systematic size bias matters most where
  volume is the clinical quantity: RANO 2.0 assesses enhancing disease, and its optional volumetric
  thresholds are +40% and −65% between examinations.
- **Why do small compartments produce extreme relative errors?** A few millilitres of disagreement is a
  large fraction of a small reference volume, and the same boundary error costs a small structure more Dice.
- **What would make a volumetric comparison between two examinations trustworthy?** The same model and
  conventions at both visits, an explicit grid, a known bias, and measurement variability small compared
  with the threshold being tested.

Exercises: (1) apply the 4.8 mm shift and the 12% expansion to the 0.53 mL label-1 core alone and compare
Dice with the whole-tumor result; (2) in Figure 3C, find subjects with Dice above 0.85 and volume error
beyond ±20%. What does each tell a clinician that the other does not?

## Image compartments and biological states

An imaging label describes a region under an explicit convention. It does not identify immune-cell composition, a malignant-cell program, or neuronal activity. Notebooks 05 and 06 explore why biology needs additional measurements, while preserving the measurement checks here.